# **A/B Testing Analysis of a Marketing Campaing** 

## **Conduct a two-proportion Z-test in order to check wether exposure of users to adds leads to increase in conversions.**

If we define conversion rate for both treatment and control group as:<br/>
**Treatment group:** $p_1 = x_1 / n_1$, where $x_1$ is the number of conversions and $n_1$ - the number of participants in the treatment group.<br/> 
**Control group**: $p_2 = x_2 / n_2$, where $x_2$ is the number of conversions and $n_2$ -the number of participants in the control group.

**Null Hypothesis ($H_0$):** $p_1 = p_2$ and any increase in conversions due to treament is random. <br/>
**Alternative Hypothesis ($H_1$):** $p_1 > p_2$, so the treatment do increase conversion rate. 

In [1]:
import os
import pandas as pd

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/MarketingA_B_Testing/"

data_df = pd.read_csv(DATASETS_PATH + "marketing_AB.csv")

# Calculate sample conversion rates:
conversions = data_df.groupby("test group")["converted"].value_counts(normalize=True)

p1 = conversions["ad"][True]
p2 = conversions["psa"][True]

print("p1 = "+ str(p1))
print()
print("p2 = "+ str(p2))


p1 = 0.025546559636683747

p2 = 0.01785410644448223


In [2]:
#Calculate pooled conversion rate:
conversion = data_df["converted"].value_counts(normalize=True)

p = conversion[True]
print("p = " + str(p))

p = 0.02523886203220195


In [3]:
# Calculate number of users in each group
n_users = data_df["test group"].value_counts()

n_users_1 = n_users["ad"]
n_users_2 = n_users["psa"]

print("Number of users in treatment group: " + str(n_users_1))
print("Number of users in control group: " + str(n_users_2))

Number of users in treatment group: 564577
Number of users in control group: 23524


In [4]:
# Calculate standard error:
import numpy as np
se = np.sqrt((p*(1-p)*((1.0/n_users_1) + (1.0/n_users_2))))
print("Standard Error: " + str(se))

Standard Error: 0.0010437410649006525


In [5]:
# Calculate Z-score:
z_score = (p1 - p2)/se
print("Z-score: " + str(z_score))

Z-score: 7.3700781265454145


In [6]:
from scipy.stats import norm

# one-sided
p_value = 1 - norm.cdf(z_score)

print("P-value: " + str(p_value))

P-value: 8.526512829121202e-14


Due to very small p-value (< 0.05), there is a very strong statistical evidence for alternative hypothesis and thus the treatment does improve conversions.

## **Calculate the treatment lift and confidence interval for it.**

In [7]:
delta = p1 - p2

se = np.sqrt(
    (p1 * (1 - p1)) / n_users_1 +
    (p2 * (1 - p2)) / n_users_2
)

z = norm.ppf(0.975)  # 95% CI -> 2.5% in each tail (total 5%), using percent point function (the inverse of CDF)

lower = delta - z * se # comes from the z-score
upper = delta + z * se

print(f"Lift: {delta:.6f}")
print(f"95% CI: [{lower:.6f}, {upper:.6f}]")

Lift: 0.007692
95% CI: [0.005951, 0.009434]


The baseline conversion rate is $\approx 1.79$% (in the control group), so the relative lift of the treatment is around $43$%.

## **Check the Null Hypothesis by Bootstrapping.**

In [8]:
ad = data_df[data_df["test group"] == "ad"]["converted"].values
psa = data_df[data_df["test group"] == "psa"]["converted"].values

combined = np.concatenate([ad, psa])
n1, n2 = len(ad), len(psa)

obs_diff = ad.mean() - psa.mean()

print("Observed difference in conversion rates: " + str(obs_diff))

number_of_experiments = 10000
diffs = []

B = number_of_experiments
combined = np.array(combined)

n = len(combined)
p = n1 / n

# random assignment matrix (B x n)
mask = np.random.rand(B, n) < p

# broadcast data without tiling
group1_means = (mask * combined).sum(axis=1) / mask.sum(axis=1)
group2_means = ((~mask) * combined).sum(axis=1) / (~mask).sum(axis=1)

diffs = group1_means - group2_means

p_value = np.mean(diffs >= obs_diff)

print("P-Value: " + str(p_value))

Observed difference in conversion rates: 0.007692453192201517
P-Value: 0.0


The bootstrapping method delivers the strong evidence too, that the null hypothesis is wrong and the treatment does have effect on conversion rates. 

## **Calculate Confidence Interval for the Difference of Conversion Rates in Groups.**

In [9]:
# vectorized resampling (B x n1, B x n2)
boot_ad = np.random.choice(ad, size=(B, n1), replace=True)
boot_psa = np.random.choice(psa, size=(B, n2), replace=True)

# compute means along rows
boot_ad_means = boot_ad.mean(axis=1)
boot_psa_means = boot_psa.mean(axis=1)

# bootstrap distribution of differences
boot_diffs = boot_ad_means - boot_psa_means

# confidence interval
lower = np.percentile(boot_diffs, 2.5)
upper = np.percentile(boot_diffs, 97.5)

print("95% CI:", (lower.item(), upper.item()))

95% CI: (0.005926437834566346, 0.009430128667592083)


The ruslt of the bootstrap calculation is very close to the confidents interval from Z-statistics above.

## **Indermidiate conclusions:**
There is a very strong statistical evidence delivered by analytical and numerical methods, that there is an effect of user treatment by adds leading to increase in conversion rate. The realtive conversion rate lift is about $43$ % laying in a $95$ % confidence interval between $33$ % and $53$ %.

## **Analysis of the effect of values of "total adds", "most ads day", "most ads hour" on conversion rate by logistic regression.**

In order to model the effect of the number of ads that users have been exposed to, we are going to conduct a logistic regression in the following form:
$$
logit(p) = \beta_0 + \beta_1*\text{ads} + \beta_2*\text{group} + \beta_3*\text{interaction} + \gamma_{day} + \sigma_{hour}
$$
Where we are basically fitting two lines corresponding to "ad" and "psa" groups in order to measure the effect of ads number in both groups and wether treatment group shows a greater effect. 

In [10]:
import statsmodels.api as sm

data_df = data_df.copy()

data_df["group"] = (data_df["test group"] == "ad").astype(int)
data_df["interaction"] = data_df["total ads"] * data_df["group"]

# one-hot encoding (VERY important: drop_first avoids multicollinearity)
day_dummies = pd.get_dummies(data_df["most ads day"], drop_first=True)
hour_dummies = pd.get_dummies(data_df["most ads hour"], drop_first=True)

X = pd.concat([
    data_df[["total ads", "group", "interaction"]],
    day_dummies,
    hour_dummies
], axis=1)

X = sm.add_constant(X)
y = data_df["converted"].astype(int)
X = X.astype(float)

model = sm.Logit(y, X)
result = model.fit(cov_type="HC3")

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.108124
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:               588101
Model:                          Logit   Df Residuals:                   588068
Method:                           MLE   Df Model:                           32
Date:                Fri, 24 Apr 2026   Pseudo R-squ.:                 0.08199
Time:                        17:27:49   Log-Likelihood:                -63588.
converged:                       True   LL-Null:                       -69267.
Covariance Type:                  HC3   LLR p-value:                     0.000
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -4.9170      0.122    -40.207      0.000      -5.157      -4.677
total ads       0.0098    

Let's explain shortly the results in the table above:
* The **LLR (Log-Liklihood-Ratio) p-value** is a test of null hypothesis that the parameters in the equation have no effect on the target log-odds variable and the value of 0.000 means that this hypothesis is wrong and the fit is statistically relevant.
* The value **Pseudo R-square** is not the same as in least squares fit, because logostic regression uses maximum liklihood estimation for fitting. The value of 0.08199 says that it is a weak but positive fit (excelent fit is for values between 0.2 and 0.4).
* **Intercept (const)** coefficient shows the baseline log-odds of conversion for "psa" group when total ads = 0. Let's convert the log-ods value back to probability:
$$
p=\frac{1}{1+e^{4.9170}} \approx 0.0073
$$
So the baseline conversion probability of around 0.7% is very low.
* **total ads** coefficient of 0.0098 means more ads shown is associated with higher conversion probability for "psa" baseline group, so each additional ad increases odds of conversion by about 1% ($exp(0.0098) \approx 1.01$)
* **group** coefficient of 0.3556 means that "ad" group performs better than "psa" group. Treatment improves baseline conversion to around 43% higher odds of conversion.
* **interaction** coefficient of 0.0005 with a p-value = 0.368 means that there is no strong evidence that the effect of ad volume differs between "ad" and "psa" groups. Since p-value is high, we do not reject the same slope for both groups. So the treatment shifts upward, but extra ads behave similarly in both groups.
* Coefficients representing **time categories**, show very strong behavioral signals, but can not be interpreted as causal.

While the A/B test provides a valid estimate of the causal effect of the advertising treatment ("ad" vs "psa"), the observed relationship between total ads and conversion cannot be interpreted causally in either group, as ad exposure is not guaranteed to be randomly assigned and is likely influenced by underlying user behavior and platform targeting mechanisms. Strong time behavioural patterns cannot be interpreted causally as well and are rather predictive. Time is not randomized, and it is entangled with user behavior, exposure, and system processes as well. The regression model estimates when conversion happens more often and not whether time causes conversion to change.

## **Convert logistic regression results into conversion lift effect by category.**

In [11]:
b = result.params

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

mean_ads = data_df["total ads"].mean()


# 1. Baseline (PSA, mean ads, no time effects)
logit_base = (
    b["const"]
    + b["total ads"] * mean_ads
)

p_base = sigmoid(logit_base)

# 2. Treatment effect
p_treat = sigmoid(logit_base + b["group"])
treatment_lift = p_treat - p_base

# 3. Ads effect (1-unit increase)
p_ads_only = sigmoid(logit_base + b["total ads"])
ads_lift = p_ads_only - p_base


# 4. Average time effect
time_cols = [c for c in b.index
             if c not in ["const", "total ads", "group", "interaction"]]

X_time = result.model.exog[:, [result.model.exog_names.index(c) for c in time_cols]]

time_effect = X_time @ b[time_cols].values
time_effect_mean = np.mean(time_effect)

p_time = sigmoid(logit_base + time_effect_mean)
time_lift = p_time - p_base

# 5. Full model (expected value over population)
# full linear predictor per observation
lin_pred = result.model.exog @ b.values
p_full = np.mean(sigmoid(lin_pred))

total_lift = p_full - p_base

# 6. Report
report = pd.DataFrame({
    "Component": [
        "Baseline",
        "Ads effect",
        "Treatment effect",
        "Time effects (avg)",
        "Total predicted lift (avg)"
    ],
    "Probability": [
        p_base,
        p_ads_only,
        p_treat,
        p_time,
        p_full
    ],
    "Lift vs baseline": [
        0,
        ads_lift,
        treatment_lift,
        time_lift,
        total_lift
    ]
})

print(report)

                    Component  Probability  Lift vs baseline
0                    Baseline     0.009252          0.000000
1                  Ads effect     0.009342          0.000090
2            Treatment effect     0.013152          0.003900
3          Time effects (avg)     0.014822          0.005570
4  Total predicted lift (avg)     0.025239          0.015987


* The baseline probability of conversion is around 0.93%. This represents a reference user in the control group, at baseline time conditions, with average ad exposure.
* The treatment effect increases the baseline conversion probability to 1.32%, which is small but positive main causal effect.
* Each additional unit of ad exposure is associated with ~1% increase in odds of conversion. In probability terms the effect is very small around +0.009 percentage points. This is not necessarily causal - exposure is not randomized and may correlate with user behavior or targeting.
* Conversion probability varies strongly depending on when users are exposed to ads (around +0.56 percentage points).

## **Conclusions:**
* The tested treatment has a strong and statistically significant positive effect, increasing conversion by approximately 43% in odds.
* Conversion is strongly influenced by time-of-day and weekday patterns, indicating that user engagement is highly cyclical and time-dependent.
* Ad exposure shows a small positive association with conversion, but this effect should be interpreted as correlational rather than causal.
* The full model suggests that combining treatment, exposure, and temporal structure yields a higher predicted conversion rate (~2.52%), but this should be interpreted as a predictive scenario, not a causal decomposition.
* In order to check an effect of ads frequency, or time impact on conversions, this variables should be set for different groups and not measured during the experiment of total effect of treatment with ads.